In [1]:
import tensorflow as tf
import cv2
import numpy as np
import albumentations as A
from sklearn.model_selection import train_test_split

print("TensorFlow version:", tf.__version__)
print("GPU tersedia:", tf.config.list_physical_devices('GPU'))

# Cek MobileNetV2 bisa diload
model = tf.keras.applications.MobileNetV2(weights='imagenet')
print("MobileNetV2 berhasil diload!")

TensorFlow version: 2.21.0
GPU tersedia: []
MobileNetV2 berhasil diload!


## 1. Load Dataset\nMenggunakan `image_dataset_from_directory` karena data (dan augmentasinya) sudah tersedia di folder masing-masing.

In [2]:
import os

dataset_dir = r"c:\Users\Tristan\Downloads\cleaned_dataset"
train_dir = os.path.join(dataset_dir, 'train')
val_dir = os.path.join(dataset_dir, 'val')
test_dir = os.path.join(dataset_dir, 'test')

BATCH_SIZE = 32
IMG_SIZE = (224, 224)

train_dataset = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    shuffle=True,
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE
)

val_dataset = tf.keras.utils.image_dataset_from_directory(
    val_dir,
    shuffle=True,
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE
)

test_dataset = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    shuffle=False,
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE
)

class_names = train_dataset.class_names
print("Class names:", class_names)


Found 18000 files belonging to 6 classes.
Found 2040 files belonging to 6 classes.
Found 2040 files belonging to 6 classes.
Class names: ['freshapples', 'freshbanana', 'freshoranges', 'rottenapples', 'rottenbanana', 'rottenoranges']


## 2. Prefetching & Preprocessing\nMeningkatkan performa I/O dataset dan menerapkan preprocessing bawaan MobileNetV2.

In [3]:
AUTOTUNE = tf.data.AUTOTUNE

train_dataset = train_dataset.prefetch(buffer_size=AUTOTUNE)
val_dataset = val_dataset.prefetch(buffer_size=AUTOTUNE)
test_dataset = test_dataset.prefetch(buffer_size=AUTOTUNE)

preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input


## 3. Membangun Arsitektur Model

In [4]:
# Buat base model MobileNetV2
base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights='imagenet'
)

# Freeze base model (kita tidak melatih layer dasar terlebih dahulu)
base_model.trainable = False

# Layer Arsitektur Klasifikasi Kustom
inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = preprocess_input(inputs) # Preprocessing spesifik MobileNetV2
x = base_model(x, training=False) # Pastikan BatchNorm layer jalan di inference mode
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(6, activation='softmax')(x) # 6 kelas buah

model = tf.keras.Model(inputs, outputs)

model.summary()


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 6s 1us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 6)              │         7,686 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,265,670 (8.64 MB)

 Trainable params: 7,686 (30.02 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

## 4. Kompilasi & Pelatihan Model

In [5]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

# Setup Callbacks
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint('best_mobilenet_model.h5', monitor='val_accuracy', save_best_only=True)
]

# Jalankan training
initial_epochs = 10

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=initial_epochs,
    callbacks=callbacks
)


Epoch 1/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 0s 338ms/step - accuracy: 0.5504 - loss: 1.2167

563/563 ━━━━━━━━━━━━━━━━━━━━ 219s 383ms/step - accuracy: 0.7214 - loss: 0.8171 - val_accuracy: 0.9387 - val_loss: 0.3011
Epoch 2/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 0s 240ms/step - accuracy: 0.8868 - loss: 0.3773

563/563 ━━━━━━━━━━━━━━━━━━━━ 149s 265ms/step - accuracy: 0.8960 - loss: 0.3394 - val_accuracy: 0.9647 - val_loss: 0.1770
Epoch 3/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 0s 325ms/step - accuracy: 0.9155 - loss: 0.2681

563/563 ━━━━━━━━━━━━━━━━━━━━ 206s 365ms/step - accuracy: 0.9201 - loss: 0.2547 - val_accuracy: 0.9730 - val_loss: 0.1306
Epoch 4/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 0s 331ms/step - accuracy: 0.9301 - loss: 0.2217

563/563 ━━━━━━━━━━━━━━━━━━━━ 206s 366ms/step - accuracy: 0.9327 - loss: 0.2112 - val_accuracy: 0.9789 - val_loss: 0.1063
Epoch 5/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 0s 312ms/step - accuracy: 0.9392 - loss: 0.1917

563/563 ━━━━━━━━━━━━━━━━━━━━ 198s 352ms/step - accuracy: 0.9405 - loss: 0.1852 - val_accuracy: 0.9809 - val_loss: 0.0915
Epoch 6/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 0s 308ms/step - accuracy: 0.9439 - loss: 0.1730

563/563 ━━━━━━━━━━━━━━━━━━━━ 197s 344ms/step - accuracy: 0.9451 - loss: 0.1682 - val_accuracy: 0.9833 - val_loss: 0.0814
Epoch 7/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 0s 294ms/step - accuracy: 0.9504 - loss: 0.1593

563/563 ━━━━━━━━━━━━━━━━━━━━ 183s 324ms/step - accuracy: 0.9510 - loss: 0.1541 - val_accuracy: 0.9848 - val_loss: 0.0738
Epoch 8/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 194s 309ms/step - accuracy: 0.9561 - loss: 0.1421 - val_accuracy: 0.9843 - val_loss: 0.0676
Epoch 9/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 169s 299ms/step - accuracy: 0.9591 - loss: 0.1323 - val_accuracy: 0.9848 - val_loss: 0.0628
Epoch 10/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 0s 233ms/step - accuracy: 0.9571 - loss: 0.1291

563/563 ━━━━━━━━━━━━━━━━━━━━ 142s 253ms/step - accuracy: 0.9586 - loss: 0.1256 - val_accuracy: 0.9853 - val_loss: 0.0581


## 5. Evaluasi & Ekspor ke .h5

In [6]:
# Evaluasi di data testing
loss, accuracy = model.evaluate(test_dataset)
print(f'Test accuracy: {accuracy:.4f}')

# Simpan model untuk dikonversi ke TFJS
model_save_path = 'mobilenet_fruit_model.h5'
model.save(model_save_path)
print(f"Model berhasil disimpan di: {model_save_path}")


64/64 ━━━━━━━━━━━━━━━━━━━━ 11s 174ms/step - accuracy: 0.9877 - loss: 0.0494


Test accuracy: 0.9877
Model berhasil disimpan di: mobilenet_fruit_model.h5
